# Section 2: Data Preparation & Feature Engineering  --> Chapter 8: Feature Set Finalization

---
---

**Book: Applied Machine Learning for Data Science Practitioners**
<BR>

**Author:** Vidya Subramanian (https://www.linkedin.com/in/vidyas/)

**Note:** Code is only included here for the steps that require it. Please refer to the book for the complete set of steps related to the goals covered in the chapter.

---
---

---

☑ **Install libraries**

---

In [ ]:
# In case you need to install the relevent packages, please uncomment lines below and run (once only)

# !pip install scikit-learn==1.2.2
# !pip install pandas==2.0.3
# !pip install numpy==1.25.2
# !pip install matplotlib==3.7.1
# !pip install pydrive2==1.6.3
# !pip install IPython==7.34.0
# !pip install seaborn==0.13.1
# !pip install scipy==1.11.4
# !pip install google.colab
# !pip install pydrive2==1.6.3
# !pip install oauth2client==4.1.3

---

☑ **Import libraries**

---

In [ ]:
# Set some common environment variables and Imports
import numpy as np
import pandas as pd  # Import pandas module for data manipulation
import warnings  # Warnings
from IPython.core.display import display, HTML  # Make the Jupyter chunk window wider
import sys

from sklearn.feature_selection import VarianceThreshold, SelectKBest, SelectPercentile, RFE
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif, r_regression, f_regression, mutual_info_regression
from sklearn.manifold import spectral_embedding
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import statsmodels.api as sm
from scipy.stats import pearsonr
from itertools import combinations

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)  # Setting to display all columns in a single row
display(HTML("<style>.container { width:100% !important; }</style>")) # Make the Jupyter chunk window wider

from pydrive2.auth import GoogleAuth  # Import GoogleAuth class from the pydrive2.auth module
from pydrive2.drive import GoogleDrive  # Import GoogleDrive class from the pydrive2.drive module
from google.colab import auth  # Import auth class from the google.colab module
from oauth2client.client import GoogleCredentials  # Import GoogleCredentials class from the oauth2client.client module

In [ ]:
# ***** Function to help display*****
def print_pretty_header(title, subtitle):
    # Define header formatting
    line_length = 50
    header_padding = 2
    # Calculate the width for the title text
    title_width = line_length  * header_padding
    print("#" * title_width + "\n")
    print(title.center(title_width) + "\n")
    if subtitle:
        # Calculate the width for the subtitle text
        subtitle_width = line_length - 2 * header_padding
        print(subtitle.center(title_width)+ "\n")
    print("#" * title_width + "\n")

In [ ]:
def get_data(colabFileName, fileLink):

    # Authenticate and create the PyDrive client.
    auth.authenticate_user()  # Authenticate the user
    gauth = GoogleAuth()  # Create a GoogleAuth instance
    gauth.credentials = GoogleCredentials.get_application_default()  # Use the default application credentials
    drive = GoogleDrive(gauth)  # Create a GoogleDrive instance using the authenticated GoogleAuth instance

    # Read File
    ignorestr, id = fileLink.split('=')  # Split the link by '=' and get the id of the file
    downloaded = drive.CreateFile({'id':id})  # Create a PyDrive File instance with the specified file id
    downloaded.GetContentFile(colabFileName)  # Download the file and save it to the local file system
    df = pd.read_csv(colabFileName)  # Read the CSV file into a Pandas dataframe
    return df  # Return the Pandas dataframe

# Read the file and display contents.
print_pretty_header("Finalize Feature Set", "All Data")
colabFileName = 'S2_Ch6_Data_Quality_Engineering_data.csv'
dataLink = 'https://drive.google.com/open?id=1h_Akm5JR06mz0yeM5jmhTfbOBDJngifp'  # The shareable link to the file
df_rawdata = get_data(colabFileName, dataLink)  # Call the get_data() function
df_rawdata_org = df_rawdata.copy()  # Make a copy of the Pandas dataframe for ease of dropping columns
df_rawdata  # Display the Pandas dataframe containing the data.


####################################################################################################

                                        Finalize Feature Set                                        

                                              All Data                                              

####################################################################################################



,VacationHomeID,VacHomeClass,VacHomeZone,VacHomeLotFrontage,VacHomeLotSqFt,VacHomeStreet,VacHomeRoad,VacHomeLotShape,VacHomeLandContour,VacHomeUtilities,VacHomeLotConfig,VacHomeLandSlope,VacHomeNeighborhood,VacHomeCondition,VacHomeBuilding,VacHomeHouseStyle,VacHomeRating,VacHomeQuality,VacHomeConsYear,VacHomeSince,VacHomeRoofStyle,VacHomeRoofMat,VacHomeExterior,VacHomeMasonryVeneer,VacHomeMasonryArea,VacHomeExterQual,VacHomeQual,VacHomeFoundation,VacHomeBsmtQuality,VacHomeBsmtLight,VacHomeBsmtFinish,VacHomeBsmtSqFt,VacHomeHeating,VacHomeHeatingQuality,VacHomeAC,VacHomeElectricalWiring,VacHomeFloor1SqFt,VacHomeFloor2SqFt,VacHomeSqFt,VacHomeNumFullBath,VacHomeNumHalfBath,VacHomeBedroomWithCloset,VacHomeKitcheninHome,VacHomeKitchenQuality,VacHomeRooms,VacHomeFireplaces,VacHomeFireplaceQuality,VacHomeGarageType,VacHomeGarageFinish,VacHomeCarsInGarage,VacHomeGarageArea,VacHomeGarageQuality,VacHomeDriveway,VacHomeWoodDeckSqFt,VacHomePorchSqFt,VacHomePoolSqFt,VacHomePoolQuality,VacHomeFence,VacHomeStartMonth,VacHomeStartYear,VacHomeSaleType,VacHomeSaleCondition,VacHomeSalePrice,VacHomeInterestInHome,VacHomeGoodSchools,VacHomeAvailableDate,VacHomeBarCode,VacHomeOwnerAddress,VacHomeOwnerCity,VacHomeOwnerCountry,VacHomeOwnerEmail,VacHomeOwnerGender,VacHomeOwnerState,VacHomeOwnerZipcode,VacHomeRenovationAmount,VacHomeSurveyDate,VacHomeSurveyRating,VacHomeReviewDate,VacHomeReviewRating
0,1,VacHomeClass-6,VacHomeZone-1,65.0,8450,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 6,Proximity to School,Apartment,Bungalow,7,5,2008,2009,Gable,Rolled Roofing,Board Batten,Natural Stone,196.0,4,3,Individual footing,4.0,No Sunlight,ZenWall Panels,856,Furnace,5,Yes,NM Cable,856,854,1710,2,1,3,1,4,8,0,NaN,Attached,Wood Sheathing,2,548,3,Paved,0,0,0,0,No Fence,2,2008,Down payment assistance,Regular,173061.00,Y,Y,2021-11-24,78408012,Jochen-Peukert-Straße 4/7\n93824 Wolgast,Schwandorf,Venezuela,sylvester59@haase.de,M,Niedersachsen,41157,93.0,2021-11-24,4.6,2021-11-24,4.6
1,2,VacHomeClass-1,VacHomeZone-1,80.0,9600,Cobblestone,Gravel,Pie,Flat,Trash|Gas|Water|Electricity,T-intersection,No Slope,RNH 25,Proximity to Hospital,Apartment,Contemporary,6,8,1977,1979,Gable,Rolled Roofing,Log Wood,Manmade Stone,0.0,3,3,Combined footing,4.0,French Doors,BrightWall Paneling,1262,Furnace,5,Yes,NM Cable,1262,0,1262,2,0,3,1,3,6,1,TA,Attached,Wood Sheathing,2,460,3,Paved,298,0,0,0,No Fence,5,2007,Down payment assistance,Regular,150651.00,N,Y,2021-10-03,23209473,Henkweg 1\n93328 Bremen,Hersbruck,Bolivien,tilman19@junck.org,F,Thüringen,84185,9.0,2021-01-16,3.8,2021-01-16,3.8
2,3,VacHomeClass-6,VacHomeZone-1,68.0,11250,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,Interior lot,No Slope,RNH 6,Proximity to School,Apartment,Bungalow,7,5,2006,2008,Gable,Rolled Roofing,Board Batten,Natural Stone,162.0,4,3,Individual footing,4.0,Small Windows,ZenWall Panels,920,Furnace,5,Yes,NM Cable,920,866,1786,2,1,3,1,4,6,1,TA,Attached,Wood Sheathing,2,608,3,Paved,0,0,0,0,No Fence,9,2008,Down payment assistance,Regular,185511.00,Y,N,2021-09-21,12309894,Mercedes-Butte-Allee 8/4\n39694 Neustrelitz,Borken,Pakistan,erich35@kade.com,M,Rheinland-Pfalz,79533,88.0,2021-07-08,4.4,2021-07-08,4.4
3,4,VacHomeClass-7,VacHomeZone-1,60.0,9550,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,Corner lot,No Slope,RNH 7,Proximity to School,Apartment,Bungalow,7,5,1916,1975,Gable,Rolled Roofing,Wood Shingle,Manmade Stone,0.0,3,3,Strip foundation,3.0,No Sunlight,BrightWall Paneling,756,Furnace,4,Yes,NM Cable,961,756,1717,1,0,3,1,4,7,1,Gd,Detached,Metal Panels,3,642,3,Paved,0,0,0,0,No Fence,2,2006,Down payment assistance,Shortsale,116206.00,Y,N,2021-01-11,9163454,Remo-Conradi-Ring 090\n21730 Potsdam,Schwarzenberg,Thailand,burkardpaertzelt@googlemail.com,F,Brandenburg,13145,94.0,2021-07-05,3.9,2021-07-05,3.9
4,5,VacHomeClass-6,VacHomeZone-1,84.0,14260,Cobblestone,Gravel,Reverse Pie,Flat,Trash|Gas|Water|Electricity,T-intersection,No Slope,RNH 14,Proximit

In [ ]:
# Calculate the number of rows needed for the training set (50%)
training_size = len(df_rawdata) // 2

# Slice the data to create the training and test sets
df_train_data = df_rawdata.iloc[:training_size]
df_test_data = df_rawdata.iloc[training_size:]

# Print the shapes of the resulting dataframes
print_pretty_header("Splitting data for Generalization", "Shape of Bifurcated datasets")
print(f"Train Data Shape: {df_train_data.shape}")
print(f"Test Data Shape: {df_test_data.shape}")

####################################################################################################

                                 Splitting data for Generalization                                  

                                    Shape of Bifurcated datasets                                    

####################################################################################################

Train Data Shape: (730, 79)
Test Data Shape: (730, 79)


# 1.1. Feature Selection

## 1.1.1. Filter Methods

### a. Feature Elimination

In [ ]:
# Feature Elimination based on Missing Value Ratio
def remove_missing_value_features(data, threshold=0.15):
    # Calculate the percentage of missing values for each feature
    for col in data.columns:
        pct_missing = data[col].isnull().sum()/ len(data[col].index)

        # Drop the feature with the highest missing value ratio, if any
        if round(pct_missing*100) > threshold:
            # print("Dropping column:", col)
            data = data.drop(columns=col)
    return data

# Feature Elimination based on Variance Threshold
def remove_low_variance_features(df_train_data, threshold=0.1):
    # Set an absolute variance threshold
    threshold = 5

    # Calculate variance for each column and drop columns with low variance
    columns_to_drop = []
    for column in df_train_data.columns:
        column_variance = df_train_data_variance[column].var()
        if column_variance < threshold:
            columns_to_drop.append(column)

    data_2 = df_train_data_variance.drop(columns=columns_to_drop)
    return data_2

# Feature Elimination based on Quasi-Constants
def remove_quasi_constants(data, threshold=5):
    quasi_constants = data.std() < threshold
    if quasi_constants.any():
        column_to_drop = quasi_constants[quasi_constants].index[0]
        print("Dropping column:", column_to_drop)
        return data.drop(columns=column_to_drop)
    else:
        return data

# Main code
# Create a DataFrame containing variables. I am taking a few variables so that the code runs faster and since this serves only as an example.
target_variable = df_train_data.VacHomeSalePrice

# Remove missing value features - I am intentionally taking examples to show removal of columns
df_train_data_missing = df_train_data[['VacHomeLotFrontage', 'VacHomeLotSqFt', 'VacHomeFireplaceQuality']]
data_1 = remove_missing_value_features(df_train_data_missing)

# Print Header
print_pretty_header("Finalize Feature Set", "Remove missing value features")
print("Columns : Before Removing missing value features:", df_train_data_missing.columns)
print("Columns : After Removing missing value features: ", data_1.columns)
print("\n")

# Remove low variance features - I am intentionally taking examples to show removal of columns
df_train_data_variance = df_train_data[['VacHomeQuality', 'VacHomeLotSqFt', 'VacHomeRating']]
data_2 = remove_low_variance_features(df_train_data_variance)

# Print Header
print_pretty_header("Finalize Feature Set", "Remove low variance features")
print("Columns : Before Removing low variance features:", df_train_data_variance.columns)
print("Columns : After Removing low variance features: ", data_2.columns)
print("\n")

# Remove quasi-constants - I am intentionally taking examples to show removal of columns
df_train_data_quasi = df_train_data[['VacHomeSurveyRating', 'VacHomeLotSqFt', 'VacHomeRating']]
data_3 = remove_quasi_constants(df_train_data_quasi)
# Print Header
print_pretty_header("Finalize Feature Set", "Remove Features with quasi-constants")
print("Columns : Before Removing Features with quasi-constants:", df_train_data_quasi.columns)
print("Columns : After Removing Features with quasi-constants: ", data_3.columns)
print("\n")


####################################################################################################

                                        Finalize Feature Set                                        

                                   Remove missing value features                                    

####################################################################################################

Columns : Before Removing missing value features: Index(['VacHomeLotFrontage', 'VacHomeLotSqFt', 'VacHomeFireplaceQuality'], dtype='object')
Columns : After Removing missing value features:  Index(['VacHomeLotSqFt'], dtype='object')


####################################################################################################

                                        Finalize Feature Set                                        

                                    Remove low variance features                                    

###################################################

### b. Univariate Filter Methods

In [ ]:
# Feature Selection using Univariate Filter Methods
def univariate_filter_methods(df_train_data_subset, target_variable, score_func_var, k=2):
    selector = SelectKBest(score_func=score_func_var, k=k)
    selected_features = selector.fit_transform(df_train_data_subset, target_variable)
    return pd.DataFrame(selected_features, columns=df_train_data_subset.columns[selector.get_support()])

# Feature Selection using SelectPercentile
def select_percentile_filter(df_train_data_subset, target_variable, score_func_var, percentile=10):
    selector = SelectPercentile(score_func=score_func_var, percentile=percentile)
    selected_features = selector.fit_transform(df_train_data_subset, target_variable)
    return pd.DataFrame(selected_features, columns=df_train_data_subset.columns[selector.get_support()])

# Example
if __name__ == '__main__':

    target_variable_regression = df_train_data.VacHomeSalePrice
    target_variable_classification = df_train_data.VacHomeSaleType

    # Remove missing value features
    df_train_data_univariate = df_train_data[['VacHomeNumFullBath', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt']]

    selected_features_r_regression = univariate_filter_methods(df_train_data_univariate, target_variable_regression, r_regression, k=2)
    selected_features_f_regression = univariate_filter_methods(df_train_data_univariate, target_variable_regression, f_regression, k=2)
    selected_features_mutual_info_regression = univariate_filter_methods(df_train_data_univariate, target_variable_regression, mutual_info_regression, k=2)

    selected_features_select_percentile_f_regression = select_percentile_filter(df_train_data_univariate, target_variable_regression, f_regression, percentile=10)

    # Print Header
    print_pretty_header("Finalize Feature Set", "Univariate Filter Methods")

    print("Columns : Before using Univariate Filter Methods : ", df_train_data_univariate.columns)
    print("\n")
    print("Columns : After using Univariate Filter Method : Regression- r_regression : ", selected_features_r_regression.columns)
    print("Columns : After using Univariate Filter Method : Regression- f_regression): ", selected_features_f_regression.columns)
    print("Columns : After using Univariate Filter Method : Regression- mutual_info_regression : ", selected_features_mutual_info_regression.columns)
    print("Columns : After using Univariate Filter Method : Regression- SelectPercentile - f_regression : ", selected_features_select_percentile_f_regression.columns)
    print("\n")

    selected_features_chi2 = univariate_filter_methods(df_train_data_univariate, target_variable_classification, chi2, k=2)
    selected_features_f_classif = univariate_filter_methods(df_train_data_univariate, target_variable_classification, f_classif, k=2)
    selected_features_mutual_info_classif = univariate_filter_methods(df_train_data_univariate, target_variable_classification, mutual_info_classif, k=2)
    print("Columns : After using Univariate Filter Method : Classification- chi2 : ", selected_features_chi2.columns)
    print("Columns : After using Univariate Filter Method : Classification- f_classif : ", selected_features_f_classif.columns)
    print("Columns : After using Univariate Filter Method : Classification- mutual_info_classif : ", selected_features_mutual_info_classif.columns)
    print("\n")

####################################################################################################

                                        Finalize Feature Set                                        

                                     Univariate Filter Methods                                      

####################################################################################################

Columns : Before using Univariate Filter Methods :  Index(['VacHomeNumFullBath', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt'], dtype='object')


Columns : After using Univariate Filter Method : Regression- r_regression :  Index(['VacHomeNumFullBath', 'VacHomeBsmtSqFt'], dtype='object')
Columns : After using Univariate Filter Method : Regression- f_regression):  Index(['VacHomeNumFullBath', 'VacHomeBsmtSqFt'], dtype='object')
Columns : After using Univariate Filter Method : Regression- mutual_info_regression :  Index(['VacHomeNumFullBath', 'VacHomeBsmtSqFt'], dtype='object')
Columns : After usi

### c. Bivariate Filter Methods

In [ ]:
# Placeholder for bivariate_filter_methods function
def bivariate_filter_methods(df, target):
    # Perform calculations for correlations and fisher_scores
    correlations = df.corrwith(target)
    fisher_scores, _ = f_regression(df, target)

    return correlations, fisher_scores

if __name__ == '__main__':
    # Drop rows with missing values
    df_train_data_bivariate_all = df_train_data[['VacHomeLotFrontage', 'VacHomeLotSqFt', 'VacHomeSalePrice']].dropna()

    df_train_data_bivariate = df_train_data_bivariate_all[['VacHomeLotFrontage', 'VacHomeLotSqFt']]
    target_variable = df_train_data_bivariate_all.VacHomeSalePrice.astype(int)

    correlations, fisher_scores = bivariate_filter_methods(df_train_data_bivariate, target_variable)

    # Create a DataFrame to store the feature selection scores
    scores_data = {
        "Feature": df_train_data_bivariate.columns,
        "Correlation Coefficient": correlations,
        "Fisher Score": fisher_scores,
    }

    scores_df = pd.DataFrame(scores_data)

    # Set thresholds for each score
    correlation_threshold = 0.3  # Example threshold
    fisher_threshold = 0.2       # Example threshold

    # Drop features based on the thresholds
    selected_features_cc = scores_df[scores_df["Correlation Coefficient"] > correlation_threshold]
    selected_features_fisher = scores_df[scores_df["Fisher Score"] > fisher_threshold]

    # Print Header
    print_pretty_header("Finalize Feature Set", "Remove Features with Bivariate Filter Methods")
    print("Columns : Before using Bivariate Filter Methods : ", df_train_data_bivariate.columns.tolist())
    print("\n")
    print("Columns : After using Bivariate Filter Method for Regression - Fisher Score : ", selected_features_fisher["Feature"].tolist())
    print("\n")
    print("Columns : After using Bivariate Filter Method for Classification - Correlation Coefficient : ", selected_features_cc["Feature"].tolist())


####################################################################################################

                                        Finalize Feature Set                                        

                           Remove Features with Bivariate Filter Methods                            

####################################################################################################

Columns : Before using Bivariate Filter Methods :  ['VacHomeLotFrontage', 'VacHomeLotSqFt']


Columns : After using Bivariate Filter Method for Regression - Fisher Score :  ['VacHomeLotFrontage', 'VacHomeLotSqFt']


Columns : After using Bivariate Filter Method for Classification - Correlation Coefficient :  ['VacHomeLotFrontage', 'VacHomeLotSqFt']


## 1.1.2. Wrapper Methods

### a. Stepwise Forward Selection

In [ ]:
# Stepwise Forward Selection
def forward_selection(data, response_col, max_features=3):
    remaining_features = set(data.columns) - {response_col}
    selected_features = []
    best_model = None
    best_aic = np.inf

    while remaining_features and len(selected_features) < max_features:
        candidate_models = []

        for feature in remaining_features:
            features_to_include = selected_features + [feature]
            X = data[features_to_include]
            X = sm.add_constant(X)
            model = sm.OLS(data[response_col], X).fit()
            candidate_models.append((model.aic, model))

        candidate_models.sort(key=lambda x: x[0])
        best_candidate_aic, best_candidate_model = candidate_models[0]

        if best_candidate_aic < best_aic:
            best_aic = best_candidate_aic
            best_model = best_candidate_model
            selected_features = [feature for feature in best_candidate_model.params.index if feature != 'const']
            remaining_features = remaining_features - set(selected_features)
        else:
            break

    return best_model, selected_features

# Example usage
response_col = "VacHomeSalePrice"
max_features = 2
df_train_data_forward_selection = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt','VacHomeSalePrice']]
target_variable = df_train_data.VacHomeSalePrice.astype(int)

# Forward Selection
forward_model, forward_selected = forward_selection(df_train_data_forward_selection, response_col, max_features)
# Print Header
print_pretty_header("Finalize Feature Set", "Stepwise Forward Selection")
print("Columns : Before using Stepwise Forward Selection : ", df_train_data_forward_selection.columns)
print("Columns : After using Stepwise Forward Selection : ",  forward_selected)

####################################################################################################

                                        Finalize Feature Set                                        

                                     Stepwise Forward Selection                                     

####################################################################################################

Columns : Before using Stepwise Forward Selection :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeSalePrice'], dtype='object')
Columns : After using Stepwise Forward Selection :  ['VacHomeSqFt', 'VacHomeBsmtSqFt']


### b. Stepwise Backward Selection

In [ ]:
# Stepwise Backward Elimination
def backward_elimination(data, response_col):
    features = set(data.columns) - {response_col}
    selected_features = list(features)
    best_model = None
    best_aic = np.inf

    while len(selected_features) > 1:
        candidate_models = []

        for feature in selected_features:
            features_to_drop = [f for f in selected_features if f != feature]
            X = data[features_to_drop]
            X = sm.add_constant(X)
            model = sm.OLS(data[response_col], X).fit()
            candidate_models.append((model.aic, model, feature))

        candidate_models.sort(key=lambda x: x[0])
        best_candidate_aic, best_candidate_model, feature_to_drop = candidate_models[0]

        if best_candidate_aic < best_aic:
            best_aic = best_candidate_aic
            best_model = best_candidate_model
            selected_features.remove(feature_to_drop)
        else:
            break

    return best_model, selected_features

# Example usage
response_col = "VacHomeSalePrice"
max_features = 2
df_train_data_backward_elimination = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt','VacHomeSalePrice']]

# Backward Elimination
backward_model, backward_selected = backward_elimination(df_train_data_backward_elimination, response_col)
# Print Header
print_pretty_header("Finalize Feature Set", "Stepwise Backward Elimination")
print("Columns : Before using Stepwise Backward Selection : ", df_train_data_backward_elimination.columns)
print("Columns : After using Stepwise Backward Selection : ",  backward_selected)

####################################################################################################

                                        Finalize Feature Set                                        

                                   Stepwise Backward Elimination                                    

####################################################################################################

Columns : Before using Stepwise Backward Selection :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeSalePrice'], dtype='object')
Columns : After using Stepwise Backward Selection :  ['VacHomeSqFt', 'VacHomeBsmtSqFt']


### c. Stepwise Regression (Combining Forward and Backward)

In [ ]:
# Stepwise Regression (Combining Forward and Backward)
def stepwise_regression(data_stepwise_reg, response_col, max_features=3):
    # Perform forward selection to build a model
    forward_model, forward_selected = forward_selection(data_stepwise_reg, response_col, max_features)

    # Perform backward elimination to build another model
    backward_model, backward_selected = backward_elimination(data_stepwise_reg, response_col)

    # Compare the AIC (Akaike Information Criterion) of the two models
    if forward_model.aic < backward_model.aic:
        return forward_model, forward_selected
    else:
        return backward_model, backward_selected

# Example usage
response_col = "VacHomeSalePrice"
max_features = 2

# Select relevant columns from the training data
df_train_data_stepwise_reg = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeSalePrice']]

# Perform Stepwise Regression to select the best features for the model
stepwise_model, stepwise_selected = stepwise_regression(df_train_data_stepwise_reg, response_col, max_features)

# Print Header
print_pretty_header("Finalize Feature Set", "Stepwise Regression (Combining Forward and Backward)")

# Print the columns before and after applying Stepwise Regression
print("Columns : Before using Stepwise Regression (Combining Forward and Backward) : ", df_train_data_stepwise_reg.columns)
print("Columns : After using Stepwise Regression (Combining Forward and Backward) : ",  stepwise_selected)


####################################################################################################

                                        Finalize Feature Set                                        

                        Stepwise Regression (Combining Forward and Backward)                        

####################################################################################################

Columns : Before using Stepwise Regression (Combining Forward and Backward) :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeSalePrice'], dtype='object')
Columns : After using Stepwise Regression (Combining Forward and Backward) :  ['VacHomeSqFt', 'VacHomeBsmtSqFt']


### d. All Possible Regressions

In [ ]:
# Define a function for Exhaustive Feature Selection
def exhaustive_feature_selection(data_all_pos_reg, response_col):
    # Create a set of features by excluding the response column
    features = set(data_all_pos_reg.columns) - {response_col}

    # Initialize variables to keep track of the best model and AIC (Akaike Information Criterion)
    best_model = None
    best_aic = np.inf

    # Loop through different numbers of features to build models with
    for k in range(1, len(features) + 1):
        # Generate all possible combinations of 'k' features
        for combo in combinations(features, k):
            # Create the feature matrix X by selecting the current combination of features
            X = data_all_pos_reg[list(combo)]

            # Add a constant term to the feature matrix (intercept term in the regression)
            X = sm.add_constant(X)

            # Fit an Ordinary Least Squares (OLS) regression model
            model = sm.OLS(data_all_pos_reg[response_col], X).fit()

            # Check if the current model has a lower AIC than the best one found so far
            if model.aic < best_aic:
                best_aic = model.aic
                best_model = model
                best_features = list(combo)

    # Return the best model and the list of features used in it
    return best_model, best_features

# Example usage
response_col = "VacHomeSalePrice"
max_features = 2

# Select a subset of columns from the training data
df_train_data_all_pos_reg = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt','VacHomeSalePrice']]

# Apply the exhaustive feature selection method
exhaustive_model, exhaustive_selected = exhaustive_feature_selection(df_train_data_all_pos_reg, response_col)

# Print Header
print_pretty_header("Finalize Feature Set", "All Possible Regressions (Exhaustive Feature Selection")

# Print the columns before and after feature selection
print("Columns : Before using All Possible Regressions : ", df_train_data_all_pos_reg.columns)
print("Columns : After using All Possible Regressions : ",  exhaustive_selected)


####################################################################################################

                                        Finalize Feature Set                                        

                       All Possible Regressions (Exhaustive Feature Selection                       

####################################################################################################

Columns : Before using All Possible Regressions :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeSalePrice'], dtype='object')
Columns : After using All Possible Regressions :  ['VacHomeLotSqFt', 'VacHomeSqFt', 'VacHomeBsmtSqFt']


### e. Recursive Feature Elimination

In [ ]:
# Separate the features and target variable
df_train_data_rfe = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt']]
target_variable = df_train_data.VacHomeSalePrice.astype(int)

# Create a linear regression model
model = LinearRegression()

# Create an RFE (Recursive Feature Elimination) object with the linear regression model and the number of features to select
rfe = RFE(model, n_features_to_select=2)

# Fit the RFE object to the data, selecting the best 2 features
rfe.fit(df_train_data_rfe, target_variable)

# Get the indices of the selected features
selected_features_idx = rfe.get_support(indices=True)

# Get the names of the selected features
selected_features = df_train_data_rfe.columns[selected_features_idx]

# Create a new DataFrame with only the selected features
selected_data = df_train_data_rfe[selected_features]

# Print Header
print_pretty_header("Finalize Feature Set", "Recursive Feature Elimination")

# Print the columns before and after applying Recursive Feature Elimination
print("Columns Before : Recursive Feature Elimination : ", df_train_data_rfe.columns)
print("Columns After : Recursive Feature Elimination : ",  selected_data.columns)


####################################################################################################

                                        Finalize Feature Set                                        

                                   Recursive Feature Elimination                                    

####################################################################################################

Columns Before : Recursive Feature Elimination :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt'], dtype='object')
Columns After : Recursive Feature Elimination :  Index(['VacHomeSqFt', 'VacHomeBsmtSqFt'], dtype='object')


## 1.1.3. Embedded Methods

### a. Lasso Cost

In [ ]:
# Select the features to be standardized
df_train_data_lasso = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt']]

# Define the target variable
target_variable = df_train_data.VacHomeSalePrice.astype(int)

# Create a StandardScaler object
scaler = StandardScaler()

# Standardize the selected features
df_train_data_lasso_std = scaler.fit_transform(df_train_data_lasso)

# Create a Lasso regression model
lasso = Lasso(alpha=0.1)
# Fit the model on the standardized data
lasso.fit(df_train_data_lasso_std, target_variable)

# Print the coefficients of the model
# print("Coefficients:", lasso.coef_)

# Print Header
print_pretty_header("Finalize Feature Set", "Filter Features using Embedded methods : Lasso Cost (L1 Regularization)")

# Get the selected features based on Lasso regularization
selected_features = df_train_data_lasso.columns[lasso.coef_ != 0]
# Print the column names before and after feature selection
print("Columns Before : Lasso Cost : ", df_train_data_lasso.columns)
print("Columns After : Lasso Cost : ", selected_data.columns)


####################################################################################################

                                        Finalize Feature Set                                        

              Filter Features using Embedded methods : Lasso Cost (L1 Regularization)               

####################################################################################################

Columns Before : Lasso Cost :  Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt'], dtype='object')
Columns After : Lasso Cost :  Index(['VacHomeSqFt', 'VacHomeBsmtSqFt'], dtype='object')


### b. Ridge Cost

In [ ]:
# Select the features to be standardized
df_train_data_ridge = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt']]

# Define the target variable
target_variable = df_train_data.VacHomeSalePrice.astype(int)

# Create a StandardScaler object
scaler = StandardScaler()
# Standardize the selected features
df_train_data_ridge_std = scaler.fit_transform(df_train_data_ridge)

# Create a Ridge regression model
ridge = Ridge(alpha=1.0)

# Fit the model on the standardized data
ridge.fit(df_train_data_ridge_std, target_variable)

# Print the coefficients of the model
# Uncomment the following line to print coefficients
# print("Coefficients:", ridge.coef_)

# Print Header
print_pretty_header("Finalize Feature Set", "Filter Features using Embedded methods: Ridge Cost (L2 Regularization).")
# Get the selected features based on Ridge regularization
selected_features = df_train_data_ridge.columns[ridge.coef_ != 0]
# Print the column names before and after feature selection
print("Columns Before: Ridge Cost:", df_train_data_ridge.columns)
print("Columns After: Ridge Cost:", selected_data.columns)


####################################################################################################

                                        Finalize Feature Set                                        

              Filter Features using Embedded methods: Ridge Cost (L2 Regularization).               

####################################################################################################

Columns Before: Ridge Cost: Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt'], dtype='object')
Columns After: Ridge Cost: Index(['VacHomeSqFt', 'VacHomeBsmtSqFt'], dtype='object')


### c. Elastic Net

In [ ]:
# Select the features to be standardized
df_train_data_elastic = df_train_data[['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt']]

# Define the target variable
target_variable = df_train_data.VacHomeSalePrice.astype(int)

# Create a StandardScaler object
scaler = StandardScaler()
# Standardize the selected features
df_train_data_elastic_std = scaler.fit_transform(df_train_data_elastic)

# Create an Elastic Net regression model
elastic_net = ElasticNet(alpha=1.0, l1_ratio=0.5)
# Fit the model on the standardized data
elastic_net.fit(df_train_data_elastic_std, target_variable)

# Print the coefficients of the model
# print("Coefficients:", elastic_net.coef_)

# Print Header
print_pretty_header("Finalize Feature Set", "Filter Features using Embedded methods: Elastic Net (L1 & L2 Regularization)")

# Get the selected features based on Elastic Net regularization
selected_features = df_train_data_elastic.columns[elastic_net.coef_ != 0]
# Print the column names before and after feature selection
print("Columns Before: Elastic Net:", df_train_data_elastic.columns)
print("Columns After: Elastic Net:", selected_data.columns)


####################################################################################################

                                        Finalize Feature Set                                        

            Filter Features using Embedded methods: Elastic Net (L1 & L2 Regularization)            

####################################################################################################

Columns Before: Elastic Net: Index(['VacHomeSqFt', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt'], dtype='object')
Columns After: Elastic Net: Index(['VacHomeSqFt', 'VacHomeBsmtSqFt'], dtype='object')


# 1.2. Dimensionality Reduction

### a. PCA

In [ ]:
# Drop the target variable for dimensionality reduction
columns_for_reduction = ['VacHomeLotFrontage', 'VacHomeLotSqFt', 'VacHomeBsmtSqFt', 'VacHomeFloor1SqFt', 'VacHomeFloor2SqFt', 'VacHomeSqFt', 'VacHomeNumFullBath', 'VacHomeNumHalfBath', \
                          'VacHomeBedroomWithCloset', 'VacHomeKitcheninHome', 'VacHomeRooms', 'VacHomeFireplaces', 'VacHomeCarsInGarage', 'VacHomeGarageArea', 'VacHomeWoodDeckSqFt', 'VacHomePorchSqFt', \
                          'VacHomePoolSqFt', 'VacHomeRenovationAmount', 'VacHomeSalePrice']
df_train_data_dim_reduction = df_train_data[columns_for_reduction]

# Drop rows with missing values
df_train_data_dim_reduction = df_train_data_dim_reduction.dropna()

# Separate the target variable
target_variable = df_train_data_dim_reduction['VacHomeSalePrice'].astype(int)
df_train_data_dim_reduction = df_train_data_dim_reduction.drop(columns=['VacHomeSalePrice'])

# Standardize the features
scaler = StandardScaler()
df_train_data_dim_reduction_std = scaler.fit_transform(df_train_data_dim_reduction)

# Apply PCA for dimensionality reduction
pca = PCA(n_components=3)  # Choose the number of components
df_pca = pca.fit_transform(df_train_data_dim_reduction_std)

# Create a DataFrame for the reduced features
columns = [f'PC{i+1}' for i in range(df_pca.shape[1])]
df_pca_with_target = pd.DataFrame(np.hstack((df_pca, target_variable.values.reshape(-1, 1))), columns=columns + ['VacHomeSalePrice'])

# Get the explained variance ratios for each component
explained_var_ratios = pca.explained_variance_ratio_

# Get the components' loadings (eigenvectors)
components_loadings = pca.components_

# Print Header
print_pretty_header("Finalize Feature Set", "Dimensionality Reduction using PCA ")

# Print the names of original columns that contribute to each principal component
for i, component_loading in enumerate(components_loadings):
    top_features_idx = np.argsort(component_loading)[::-1][:3]  # Get top 3 features for each component
    top_features = df_train_data_dim_reduction.columns[top_features_idx]
    print(f"Top features for PC{i+1}: {', '.join(top_features)} (Explained Variance: {explained_var_ratios[i]:.2f})")


####################################################################################################

                                        Finalize Feature Set                                        

                                Dimensionality Reduction using PCA                                  

####################################################################################################

Top features for PC1: VacHomeSqFt, VacHomeRooms, VacHomeCarsInGarage (Explained Variance: 0.30)
Top features for PC2: VacHomeFloor2SqFt, VacHomeBedroomWithCloset, VacHomeNumHalfBath (Explained Variance: 0.13)
Top features for PC3: VacHomeKitcheninHome, VacHomeBedroomWithCloset, VacHomeFloor1SqFt (Explained Variance: 0.08)


# 1.3. Feature Construction

In [ ]:
# Create new feature: Bedroom to Bathroom Ratio
df_train_data['BedroomToBathroomRatio'] = (df_train_data['VacHomeNumFullBath'] + 0.5 * df_train_data['VacHomeNumHalfBath']) / df_train_data['VacHomeBedroomWithCloset']

# Print selected columns including the newly created feature on a single row
columns_to_print = ['BedroomToBathroomRatio', 'VacHomeNumFullBath', 'VacHomeNumHalfBath', 'VacHomeBedroomWithCloset']

# Print Header
print_pretty_header("Finalize Feature Set", "Feature Construction and Printing")
df_train_data[columns_to_print]

####################################################################################################

                                        Finalize Feature Set                                        

                                 Feature Construction and Printing                                  

####################################################################################################



,BedroomToBathroomRatio,VacHomeNumFullBath,VacHomeNumHalfBath,VacHomeBedroomWithCloset
0,0.833333,2,1,3
1,0.666667,2,0,3
2,0.833333,2,1,3
3,0.333333,1,0,3
4,0.625000,2,1,4
...,...,...,...,...
725,0.333333,1,0,3
726,0.666667,2,0,3
727,1.000000,2,0,2
728,0.500000,2,0,4


------------- End of S2_Ch8_Finalize_Feature_Set_Code Code ------------ Vidya Subramanian ------------------